In [86]:
import pandas as pd
import numpy as np

from pathlib import Path
from sklearn.model_selection import KFold

PROJECT_ROOT = Path.cwd().parent
DATA_INTERIM = PROJECT_ROOT / "data" / "interim"

modeling_data = pd.read_csv(
    DATA_INTERIM / "modeling_data.csv",
    dtype={"FIPS": "string"}
)

print("Dataset shape:", modeling_data.shape)
print("Unique FIPS:", modeling_data["FIPS"].nunique())

Dataset shape: (3135, 22)
Unique FIPS: 3135


In [87]:
selected_predictors = [
    "PCT_LACCESS_POP19",
    "PCT_LACCESS_LOWI19",
    "GROCPTH20",
    "CONVSPTH20",
    "FFRPTH20",
    "FSRPTH20",
    "MEDHHINC21",
    "POVRATE21",
    "CHILDPOVRATE21",
    "DEEPPOVRATE21",
    "PC_SNAPBEN22",
    "PCT_65OLDER20",
    "PCT_18YOUNGER20",
    "PCT_NHWHITE20",
    "PCT_NHBLACK20",
    "PCT_HISP20",
    "PCT_NHASIAN20",
    "RECFACPTH20"
]

X = modeling_data[selected_predictors].copy()
y = modeling_data["OBESITY_AdjPrev"].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (3135, 18)
y shape: (3135,)


In [88]:
RANDOM_STATE = 42

outer_cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

In [89]:
fold_summary = []

for fold_number, (train_idx, val_idx) in enumerate(
    outer_cv.split(X),
    start=1
):
    fold_summary.append({
        "outer_fold": fold_number,
        "training_count": len(train_idx),
        "validation_count": len(val_idx)
    })

fold_summary = pd.DataFrame(fold_summary)

display(fold_summary)

print(
    "Total validation observations:",
    fold_summary["validation_count"].sum()
)

,outer_fold,training_count,validation_count
0,1,2508,627
1,2,2508,627
2,3,2508,627
3,4,2508,627
4,5,2508,627


Total validation observations: 3135


In [90]:
modeling_data["outer_fold"] = 0

for fold_number, (_, val_idx) in enumerate(
    outer_cv.split(X),
    start=1
):
    modeling_data.loc[val_idx, "outer_fold"] = fold_number

print(
    modeling_data["outer_fold"]
    .value_counts()
    .sort_index()
)

print(
    "\nCounties without fold assignment:",
    (modeling_data["outer_fold"] == 0).sum()
)

print(
    "Unique FIPS:",
    modeling_data["FIPS"].nunique()
)

outer_fold
1    627
2    627
3    627
4    627
5    627
Name: count, dtype: int64

Counties without fold assignment: 0
Unique FIPS: 3135


In [91]:
fold_1_train_idx, fold_1_val_idx = next(outer_cv.split(X))

X_train_outer = X.iloc[fold_1_train_idx].copy()
X_val_outer = X.iloc[fold_1_val_idx].copy()

y_train_outer = y.iloc[fold_1_train_idx].copy()
y_val_outer = y.iloc[fold_1_val_idx].copy()

print("X training:", X_train_outer.shape)
print("X validation:", X_val_outer.shape)
print("y training:", y_train_outer.shape)
print("y validation:", y_val_outer.shape)

X training: (2508, 18)
X validation: (627, 18)
y training: (2508,)
y validation: (627,)


In [92]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

def preprocess_outer_fold(X_train_outer, X_val_outer, missing_threshold=20.0):
    """
    Apply fold-specific preprocessing using only the outer training data.

    Order follows the thesis methodology:
    1. Missingness screening
    2. Median imputation
    3. Pearson correlation screening
    4. IQR screening
    5. Standardization for Linear Regression
    """

    # Missingness screening
    missing_summary = pd.DataFrame({
        "missing_count": X_train_outer.isna().sum(),
        "missing_pct": X_train_outer.isna().mean() * 100
    }).round(2)

    excluded_missing = missing_summary[
        missing_summary["missing_pct"] > missing_threshold
    ].index.tolist()

    retained_after_missing = [
        col for col in X_train_outer.columns
        if col not in excluded_missing
    ]

    X_train = X_train_outer[retained_after_missing].copy()
    X_val = X_val_outer[retained_after_missing].copy()

    # Median imputation
    imputer = SimpleImputer(strategy="median")
    imputer.fit(X_train)

    X_train_imputed = pd.DataFrame(
        imputer.transform(X_train),
        columns=X_train.columns,
        index=X_train.index
    )

    X_val_imputed = pd.DataFrame(
        imputer.transform(X_val),
        columns=X_val.columns,
        index=X_val.index
    )

    # Pearson correlation screening
    corr_matrix = X_train_imputed.corr(method="pearson")

    high_corr_pairs = []

    columns = corr_matrix.columns

    for i in range(len(columns)):
        for j in range(i + 1, len(columns)):
            r = corr_matrix.iloc[i, j]

            if abs(r) >= 0.80:
                high_corr_pairs.append({
                    "predictor_1": columns[i],
                    "predictor_2": columns[j],
                    "r": r,
                    "abs_r": abs(r)
                })

    high_corr_pairs = pd.DataFrame(high_corr_pairs)

    correlation_excluded = []

    if (
        "POVRATE21" in X_train_imputed.columns
        and "CHILDPOVRATE21" in X_train_imputed.columns
        and abs(
            corr_matrix.loc["POVRATE21", "CHILDPOVRATE21"]
        ) >= 0.80
    ):
        correlation_excluded.append("CHILDPOVRATE21")

    if (
        "PCT_LACCESS_POP19" in X_train_imputed.columns
        and "PCT_LACCESS_LOWI19" in X_train_imputed.columns
        and abs(
            corr_matrix.loc[
                "PCT_LACCESS_POP19",
                "PCT_LACCESS_LOWI19"
            ]
        ) >= 0.80
    ):
        correlation_excluded.append("PCT_LACCESS_POP19")

    retained_after_correlation = [
        col for col in X_train_imputed.columns
        if col not in correlation_excluded
    ]

    X_train_imputed = X_train_imputed[
        retained_after_correlation
    ].copy()

    X_val_imputed = X_val_imputed[
        retained_after_correlation
    ].copy()

    # IQR screening
    iqr_summary = []

    for column in X_train_imputed.columns:
        values = X_train_imputed[column]

        q1 = values.quantile(0.25)
        q3 = values.quantile(0.75)
        iqr = q3 - q1

        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr

        outlier_mask = (
            (values < lower_bound) |
            (values > upper_bound)
        )

        iqr_summary.append({
            "predictor": column,
            "Q1": q1,
            "Q3": q3,
            "IQR": iqr,
            "lower_bound": lower_bound,
            "upper_bound": upper_bound,
            "outlier_count": int(outlier_mask.sum()),
            "min": values.min(),
            "max": values.max()
        })

    iqr_summary = pd.DataFrame(iqr_summary)

    implausible_values_removed = 0

    # Standardization for Linear Regression-
    scaler = StandardScaler()
    scaler.fit(X_train_imputed)

    X_train_scaled = pd.DataFrame(
        scaler.transform(X_train_imputed),
        columns=X_train_imputed.columns,
        index=X_train_imputed.index
    )

    X_val_scaled = pd.DataFrame(
        scaler.transform(X_val_imputed),
        columns=X_val_imputed.columns,
        index=X_val_imputed.index
    )

    summary = {
        "excluded_missing": excluded_missing,
        "correlation_excluded": correlation_excluded,
        "retained_predictors": X_train_imputed.columns.tolist(),
        "high_corr_pairs": high_corr_pairs,
        "missing_summary": missing_summary,
        "iqr_summary": iqr_summary,
        "iqr_flagged_values": int(
            iqr_summary["outlier_count"].sum()
        ),
        "iqr_predictors_flagged": int(
            (iqr_summary["outlier_count"] > 0).sum()
        ),
        "implausible_values_removed": implausible_values_removed,
        "imputer": imputer,
        "scaler": scaler
    }

    return {
        "X_train_imputed": X_train_imputed,
        "X_val_imputed": X_val_imputed,
        "X_train_scaled": X_train_scaled,
        "X_val_scaled": X_val_scaled,
        "summary": summary
    }

In [82]:
outer_fold_results = {}
outer_fold_summaries = []

for fold_number, (train_idx, val_idx) in enumerate(
    outer_cv.split(X),
    start=1
):
    X_train_outer = X.iloc[train_idx].copy()
    X_val_outer = X.iloc[val_idx].copy()

    y_train_outer = y.iloc[train_idx].copy()
    y_val_outer = y.iloc[val_idx].copy()

    processed = preprocess_outer_fold(
        X_train_outer,
        X_val_outer,
        missing_threshold=20.0
    )

    outer_fold_results[fold_number] = {
        "X_train_imputed": processed["X_train_imputed"],
        "X_val_imputed": processed["X_val_imputed"],
        "X_train_scaled": processed["X_train_scaled"],
        "X_val_scaled": processed["X_val_scaled"],
        "y_train": y_train_outer,
        "y_val": y_val_outer,
        "summary": processed["summary"],
        "train_idx": train_idx,
        "val_idx": val_idx
    }

    summary = processed["summary"]

    outer_fold_summaries.append({
        "outer_fold": fold_number,
        "training_count": len(train_idx),
        "validation_count": len(val_idx),
        "missingness_excluded": len(summary["excluded_missing"]),
        "correlation_excluded": len(summary["correlation_excluded"]),
        "predictors_retained": len(summary["retained_predictors"]),
        "iqr_flagged_values": summary["iqr_flagged_values"],
        "iqr_predictors_flagged": summary["iqr_predictors_flagged"],
        "implausible_values_removed": summary["implausible_values_removed"]
    })

outer_fold_summaries = pd.DataFrame(outer_fold_summaries)

display(outer_fold_summaries)

,outer_fold,training_count,validation_count,missingness_excluded,correlation_excluded,predictors_retained,iqr_flagged_values,iqr_predictors_flagged,implausible_values_removed
0,1,2508,627,2,2,14,1981,14,0
1,2,2508,627,2,2,14,1957,14,0
2,3,2508,627,2,2,14,1998,14,0
3,4,2508,627,2,2,14,1988,14,0
4,5,2508,627,2,2,14,1971,14,0


In [93]:
for fold_number, result in outer_fold_results.items():

    print(f"Outer Fold {fold_number}")

    print(
        "Imputed train missing:",
        result["X_train_imputed"].isna().sum().sum()
    )

    print(
        "Imputed validation missing:",
        result["X_val_imputed"].isna().sum().sum()
    )

    print(
        "Scaled train missing:",
        result["X_train_scaled"].isna().sum().sum()
    )

    print(
        "Scaled validation missing:",
        result["X_val_scaled"].isna().sum().sum()
    )

    print(
        "Train/validation columns match:",
        result["X_train_imputed"].columns.equals(
            result["X_val_imputed"].columns
        )
    )

    print(
        "Number of predictors:",
        result["X_train_imputed"].shape[1]
    )

    print("-" * 40)

Outer Fold 1
Imputed train missing: 0
Imputed validation missing: 0
Scaled train missing: 0
Scaled validation missing: 0
Train/validation columns match: True
Number of predictors: 14
----------------------------------------
Outer Fold 2
Imputed train missing: 0
Imputed validation missing: 0
Scaled train missing: 0
Scaled validation missing: 0
Train/validation columns match: True
Number of predictors: 14
----------------------------------------
Outer Fold 3
Imputed train missing: 0
Imputed validation missing: 0
Scaled train missing: 0
Scaled validation missing: 0
Train/validation columns match: True
Number of predictors: 14
----------------------------------------
Outer Fold 4
Imputed train missing: 0
Imputed validation missing: 0
Scaled train missing: 0
Scaled validation missing: 0
Train/validation columns match: True
Number of predictors: 14
----------------------------------------
Outer Fold 5
Imputed train missing: 0
Imputed validation missing: 0
Scaled train missing: 0
Scaled valid

In [94]:
# Inner 3-fold cross-validation
INNER_SPLITS = 3

inner_cv = KFold(
    n_splits=INNER_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

print("Inner CV folds:", inner_cv.get_n_splits())

Inner CV folds: 3


In [85]:
outer_fold_1 = outer_fold_results[1]

X_outer_train_check = outer_fold_1["X_train_imputed"]

inner_fold_summary = []

for inner_fold, (inner_train_idx, inner_val_idx) in enumerate(
    inner_cv.split(X_outer_train_check),
    start=1
):
    inner_fold_summary.append({
        "inner_fold": inner_fold,
        "training_count": len(inner_train_idx),
        "validation_count": len(inner_val_idx)
    })

inner_fold_summary = pd.DataFrame(inner_fold_summary)

display(inner_fold_summary)

print(
    "Total inner validation observations:",
    inner_fold_summary["validation_count"].sum()
)

,inner_fold,training_count,validation_count
0,1,1672,836
1,2,1672,836
2,3,1672,836


Total inner validation observations: 2508
